In [87]:
import sys
import os
sys.path.append(os.path.join('..', 'dataloader'))
from dataloader_newformat import PairDatasetDisk
from preprocess import StandardizeTransform, get_min_max_coords
from dataloader_utils import gen_multistep_col_indices, levelwise_variable2, \
    delevelwise_variable, levelwise_variable
from torchvision.transforms import Compose
import json
import numpy as np

In [88]:
with open("/Users/jiandachen/Projects/NNCAM_packages/nncam_training/training_scripts/combined_vae_configs/all_varaibles.json", "r") as f:
    variable_config = json.load(f)
print(variable_config["x_ae"])

['QL_lev', 'T_nn_in_lev', 'dqvls_nn_in_lev', 'dTls_nn_in_lev', 'SOLIN', 'SPPS', 'LWUP', 'CAPE', 'UL_lev', 'VL_lev']


In [89]:
col_names_x_resmlp = variable_config["x_resmlp"]
prev_ex_vars_resmlp = variable_config["x_resmlp_prev"]
col_names_x_ae = variable_config["x_ae"]
prev_ex_vars_ae = variable_config["x_ae_prev"]
if variable_config["reduce_lvl"]:
    col_names_x_ae = levelwise_variable2(col_names_x_ae, [12,18,23,28,29])
    prev_ex_vars_ae = levelwise_variable2(prev_ex_vars_ae, [12,18,23,28,29])
else:
    col_names_x_ae = levelwise_variable(col_names_x_ae, 6, 29)
    prev_ex_vars_ae = levelwise_variable(prev_ex_vars_ae, 6, 29)
col_names_x_resmlp = delevelwise_variable(col_names_x_resmlp)
prev_ex_vars_resmlp = delevelwise_variable(prev_ex_vars_resmlp)
col_names_y = variable_config["resmlp_target"]

In [90]:
print(col_names_x_ae)

['QL_lev12', 'QL_lev18', 'QL_lev23', 'QL_lev28', 'QL_lev29', 'T_nn_in_lev12', 'T_nn_in_lev18', 'T_nn_in_lev23', 'T_nn_in_lev28', 'T_nn_in_lev29', 'dqvls_nn_in_lev12', 'dqvls_nn_in_lev18', 'dqvls_nn_in_lev23', 'dqvls_nn_in_lev28', 'dqvls_nn_in_lev29', 'dTls_nn_in_lev12', 'dTls_nn_in_lev18', 'dTls_nn_in_lev23', 'dTls_nn_in_lev28', 'dTls_nn_in_lev29', 'SOLIN', 'SPPS', 'LWUP', 'CAPE', 'UL_lev12', 'UL_lev18', 'UL_lev23', 'UL_lev28', 'UL_lev29', 'VL_lev12', 'VL_lev18', 'VL_lev23', 'VL_lev28', 'VL_lev29']


In [91]:
data_means = dict(np.load("../consts/data_means_old.npz"))
data_means_by_lvl = dict(np.load("../consts/data_means_lvl_old.npz"))
data_means.update(data_means_by_lvl)
data_stds = dict(np.load("../consts/data_stds_old.npz"))
data_stds_by_lvl = dict(np.load("../consts/data_stds_lvl_old.npz"))
data_stds.update(data_stds_by_lvl)

for k in data_means:
    if "_lev" in k and \
        ("QL" not in k and "qtend" not in k and "dqvls" not in k):
        # debug_print(f"Replacing means {k}({data_means[k]}) with\
                    #  {k.rstrip('_lev')}({data_means[k.split('_lev')[0]]})", DEBUG)
        data_means[k] = data_means[k.split("_lev")[0]]
        # debug_print(f"Replacing stds {k}({data_stds[k]}) with\
                    #  {k.rstrip('_lev')}({data_stds[k.split('_lev')[0]]})", DEBUG)
        data_stds[k] = data_stds[k.split("_lev")[0]]

In [92]:
col_names = np.loadtxt("../analysis/test_data/col_names.txt", dtype=str)
input_indices_ae, prev_input_indices_ae, output_indices_ae = gen_multistep_col_indices(
    col_names, prev_ex_vars_ae, col_names_x_ae, [], 1)
print("input_indices_ae:", col_names[input_indices_ae])
print("prev_input_indices_ae:", col_names[prev_input_indices_ae])
print("output_indices_ae:", col_names[output_indices_ae])

input_indices_resmlp, prev_input_indices_resmlp, output_indices_resmlp = gen_multistep_col_indices(
    col_names, prev_ex_vars_resmlp, col_names_x_resmlp, col_names_y, 1)
print("input_indices_resmlp:", col_names[input_indices_resmlp])
print("prev_input_indices_resmlp:", col_names[prev_input_indices_resmlp])
print("output_indices_resmlp:", col_names[output_indices_resmlp])

input_indices_ae: ['QL_lev12' 'QL_lev18' 'QL_lev23' 'QL_lev28' 'QL_lev29' 'T_nn_in_lev12'
 'T_nn_in_lev18' 'T_nn_in_lev23' 'T_nn_in_lev28' 'T_nn_in_lev29'
 'dqvls_nn_in_lev12' 'dqvls_nn_in_lev18' 'dqvls_nn_in_lev23'
 'dqvls_nn_in_lev28' 'dqvls_nn_in_lev29' 'dTls_nn_in_lev12'
 'dTls_nn_in_lev18' 'dTls_nn_in_lev23' 'dTls_nn_in_lev28'
 'dTls_nn_in_lev29' 'SOLIN' 'SPPS' 'LWUP' 'CAPE' 'UL_lev12' 'UL_lev18'
 'UL_lev23' 'UL_lev28' 'UL_lev29' 'VL_lev12' 'VL_lev18' 'VL_lev23'
 'VL_lev28' 'VL_lev29']
prev_input_indices_ae: ['QL_lev12' 'QL_lev18' 'QL_lev23' 'QL_lev28' 'QL_lev29' 'T_nn_in_lev12'
 'T_nn_in_lev18' 'T_nn_in_lev23' 'T_nn_in_lev28' 'T_nn_in_lev29'
 'dqvls_nn_in_lev12' 'dqvls_nn_in_lev18' 'dqvls_nn_in_lev23'
 'dqvls_nn_in_lev28' 'dqvls_nn_in_lev29' 'dTls_nn_in_lev12'
 'dTls_nn_in_lev18' 'dTls_nn_in_lev23' 'dTls_nn_in_lev28'
 'dTls_nn_in_lev29' 'SOLIN' 'SPPS' 'LWUP' 'CAPE' 'UL_lev12' 'UL_lev18'
 'UL_lev23' 'UL_lev28' 'UL_lev29' 'VL_lev12' 'VL_lev18' 'VL_lev23'
 'VL_lev28' 'VL_lev29' 'qte

In [93]:
multistep = 1
multistep_col_names_x_ae = []
for i in range(int(multistep)):
    multistep_col_names_x_ae.extend(col_names_x_ae)
    multistep_col_names_x_ae.extend(prev_ex_vars_ae)
multistep_col_names_x_ae.extend(col_names_x_ae)

multistep_col_names_x_resmlp = []
for i in range(int(multistep)):
    multistep_col_names_x_resmlp.extend(col_names_x_resmlp)
    multistep_col_names_x_resmlp.extend(prev_ex_vars_resmlp)
multistep_col_names_x_resmlp.extend(col_names_x_resmlp)

In [94]:
transform_ae = Compose([
        # RectRegionMaskTransform(region_mask),
        StandardizeTransform(
            data_means,
            data_stds,
            multistep_col_names_x_ae,
            [],
            col_names,
            normalize_input=True,
            normalize_output=True,
            threshold=1e10,
            ),
    ])
transform_resmlp = Compose([
    # RectRegionMaskTransform(region_mask),
    StandardizeTransform(
        data_means,
        data_stds,
        multistep_col_names_x_resmlp,
        col_names_y,
        col_names,
        normalize_input=True,
        normalize_output=True,
        include_raw=True,
        threshold=1e10,
        ),
])

In [95]:
variable_config["resmlp_input_size"][0] = len(prev_input_indices_resmlp)*int(multistep)+len(input_indices_resmlp)
variable_config["encoder"]["input_size"][0] = len(prev_input_indices_ae)*int(multistep)+len(input_indices_ae)
variable_config["decoder"]["input_size"][0] = len(prev_input_indices_ae)*int(multistep)+len(input_indices_ae)

In [96]:
print(variable_config)

{'resmlp_input_size': [526, 32, 58], 'variational': True, 'latent_size': 8, 'sample_rate': 12, 'ae_lr': 0.0001, 'resmlp_lr': 0.001, 'ae_warmup_epochs': 50, 'beta': 0.002, 'ltype': 'mse', 'reduce_lvl': True, 'resmlp_target': ['qtend_check'], 'x_ae': ['QL_lev', 'T_nn_in_lev', 'dqvls_nn_in_lev', 'dTls_nn_in_lev', 'SOLIN', 'SPPS', 'LWUP', 'CAPE', 'UL_lev', 'VL_lev'], 'x_ae_prev': ['qtend_check_lev', 'stend_check_lev', 'SOLL', 'SOLLD', 'SOLS', 'SOLSD', 'FSDS', 'CLOUD_lev', 'SPPRECC', 'FLNS', 'FLNT', 'SPQRL_lev', 'SPQRS_lev'], 'x_resmlp': ['QL_lev', 'T_nn_in_lev', 'dqvls_nn_in_lev', 'dTls_nn_in_lev', 'SOLIN', 'SPPS', 'LWUP', 'CAPE', 'UL_lev', 'VL_lev'], 'x_resmlp_prev': ['qtend_check_lev', 'stend_check_lev', 'SOLL', 'SOLLD', 'SOLS', 'SOLSD', 'FSDS', 'CLOUD_lev', 'SPPRECC', 'FLNS', 'FLNT', 'SPQRL_lev', 'SPQRS_lev'], 'encoder': {'variational': True, 'input_size': [101, 32, 58], 'channel_sizes': [512, 256, 128, 64, 32, 8], 'kernel_size': 3, 'stride': [1, 1, 1, 1, 1, 1], 'conv_padding': [1, 1, 1

In [97]:
from offline_test_ae_paired import prep_dataloaders, prep_models, prep_batchdata
import glob

In [98]:
data_dir = "../analysis/test_data"
all_files = glob.glob(data_dir + "/*.npy")

In [99]:
region_mask = np.load("../consts/twp_region.npy")

In [100]:
testing_set = PairDatasetDisk(
    all_files,
    curr_input_indices1=input_indices_ae,
    curr_input_indices2=input_indices_resmlp,
    prev_input_indices1=prev_input_indices_ae,
    prev_input_indices2=prev_input_indices_resmlp,
    output_indices1=output_indices_ae,
    output_indices2=output_indices_resmlp,
    multistep=int(multistep),
    sample_rate=1,
    is_train=False,
    transform1=transform_ae,
    transform2=transform_resmlp,
    include_filename=True,
    region_mask2d=(region_mask,2)
    )

Missing ../analysis/test_data/08688.npy for ../analysis/test_data/08689.npy
Missing ../analysis/test_data/08690.npy for ../analysis/test_data/08691.npy
Missing ../analysis/test_data/35040.npy for ../analysis/test_data/35041.npy
Missing ../analysis/test_data/35042.npy for ../analysis/test_data/35043.npy
is_train: False, dataset size: 576


In [101]:
sample = testing_set.__getitem__(0)

In [102]:
len(sample)

7

In [103]:
sample[-1]

['../analysis/test_data/08691.npy', '../analysis/test_data/08692.npy']

In [104]:
sys.path.append(os.path.join('..', 'models'))
import variational_autoencoder
import torch

In [105]:
def get_min_max_coords(mask, pad):
    min_x = np.min(np.where(mask)[0])
    max_x = np.max(np.where(mask)[0])
    min_y = np.min(np.where(mask)[1])
    max_y = np.max(np.where(mask)[1])
    # wid_x = math.ceil((max_x - min_x)/2)*2
    # wid_y = math.floor((max_y - min_y)/2)*2
    # return int(min_x-pad), int(min_x+wid_x+pad), \
    #     int(min_y-pad), int(min_y+wid_y+pad)
    #min_x, max_x, min_y, max_y =  min_x-pad, max_x+pad+1, min_y-pad, max_y+pad+1
    min_x, max_x, min_y, max_y =  min_x-pad, max_x+pad, min_y-pad, max_y+pad
    min_x = max(0, min_x)
    max_x = min(mask.shape[0], max_x)
    min_y = max(0, min_y)
    max_y = min(mask.shape[1], max_y)
    return int(min_x), int(max_x), int(min_y), int(max_y)
min_x, max_x, min_y, max_y = get_min_max_coords(region_mask, 2)
lon = np.linspace(0,357.5,144)
lat = np.linspace(-90,90,96)
print("Region window coordinates: ", lon[min_y], lon[max_y], lat[min_x], lat[max_x])
sub_region_mask = region_mask[min_x:max_x, min_y:max_y].astype(bool)

Region window coordinates:  105.0 250.0 -31.263157894736842 29.368421052631575


In [106]:
print(f"Model input size: {variable_config['resmlp_input_size']}")
    #model = autoencoder.AutoencoderResMLP(input_size, 30, args.node_size, \
    # args.activation, args.num_blocks, args.latent_dim, region_mask=region_mask, resmlp=(args.pred_weight!=0))
print(f"Loading model config: {json.dumps(variable_config, indent=4)}")
if "variational" in variable_config and variable_config["variational"] is True:
    model_struc = variational_autoencoder.SepInputAutoencoderResMLP
    variational_flag = True
else:
    raise NotImplementedError
    model_struc = autoencoder.AutoencoderResMLP
    variational_flag = False
model = model_struc(
    encoder_config=variable_config["encoder"],
    decoder_config=variable_config["decoder"],
    #input_size=len(training_set.input_indices),
    latent_size=variable_config["latent_size"],
    input_size=variable_config["resmlp_input_size"][0],
    output_size=30,
    m=512,
    activation='relu',
    num_blocks=7,
    sub_region_mask=sub_region_mask
)
print("Model structure:")
print(model)
print('Total params: %.2f' % (sum(p.numel() for p in model.parameters())))

model.float()

Model input size: [526, 32, 58]
Loading model config: {
    "resmlp_input_size": [
        526,
        32,
        58
    ],
    "variational": true,
    "latent_size": 8,
    "sample_rate": 12,
    "ae_lr": 0.0001,
    "resmlp_lr": 0.001,
    "ae_warmup_epochs": 50,
    "beta": 0.002,
    "ltype": "mse",
    "reduce_lvl": true,
    "resmlp_target": [
        "qtend_check"
    ],
    "x_ae": [
        "QL_lev",
        "T_nn_in_lev",
        "dqvls_nn_in_lev",
        "dTls_nn_in_lev",
        "SOLIN",
        "SPPS",
        "LWUP",
        "CAPE",
        "UL_lev",
        "VL_lev"
    ],
    "x_ae_prev": [
        "qtend_check_lev",
        "stend_check_lev",
        "SOLL",
        "SOLLD",
        "SOLS",
        "SOLSD",
        "FSDS",
        "CLOUD_lev",
        "SPPRECC",
        "FLNS",
        "FLNT",
        "SPQRL_lev",
        "SPQRS_lev"
    ],
    "x_resmlp": [
        "QL_lev",
        "T_nn_in_lev",
        "dqvls_nn_in_lev",
        "dTls_nn_in_lev",
        "SOLIN

SepInputAutoencoderResMLP(
  (encoder): Encoder(
    (conv): Sequential(
      (0): Conv2d(101, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): GDN(
        (beta_reparam): NonNegativeParametrizer(
          (lower_bound): LowerBound()
        )
        (gamma_reparam): NonNegativeParametrizer(
          (lower_bound): LowerBound()
        )
      )
      (2): Conv2d(512, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): GDN(
        (beta_reparam): NonNegativeParametrizer(
          (lower_bound): LowerBound()
        )
        (gamma_reparam): NonNegativeParametrizer(
          (lower_bound): LowerBound()
        )
      )
      (4): Conv2d(256, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (5): GDN(
        (beta_reparam): NonNegativeParametrizer(
          (lower_bound): LowerBound()
        )
        (gamma_reparam): NonNegativeParametrizer(
          (lower_bound): LowerBound()
        )
      )
      (6): Conv2d(128, 64, kernel

In [107]:
print(sub_region_mask.shape)

(32, 58)


In [108]:
def prep_batchdata(sample):
    x_ae, _, x_resmlp, y_resmlp, x_raw, y_raw, filenames = sample
    y_resmlp = torch.from_numpy(y_resmlp[:, sub_region_mask][None,])
    x_raw = torch.from_numpy(x_raw[:, sub_region_mask][None,])
    print(y_resmlp.shape)
    print(x_raw.shape)
    #points_y = points_y[:, :, model.sub_region_mask]
    # points_y: shape (batch, features, n_samples)
    y_resmlp = y_resmlp.permute(0, 2, 1).reshape(-1, y_resmlp.shape[1])
    x_raw = x_raw.permute(0, 2, 1).reshape(-1, x_raw.shape[1])
    print(y_resmlp.shape)
    print(x_raw.shape)
    # points_y: shape (batch*n_sample, features)
    x_ae = torch.from_numpy(x_ae[None,]).float()
    x_resmlp = torch.from_numpy(x_resmlp[None,]).float()
    # x_resmlp = x_resmlp.float().cuda()
    return x_ae, x_resmlp, y_resmlp, x_raw, y_raw, filenames

In [109]:
x_ae, x_resmlp, y_resmlp, x_raw, y_raw, filenames = prep_batchdata(sample)

IndexError: boolean index did not match indexed array along dimension 1; dimension is 33 but corresponding boolean dimension is 32

In [81]:
print(x_ae.shape)
print(x_resmlp.shape)

torch.Size([1, 101, 33, 59])
torch.Size([1, 526, 33, 59])


In [82]:
model.eval()
output = model(x_ae, x_resmlp)

In [84]:
print(output[0].shape)

torch.Size([1442, 30])


In [85]:
print(output[1].shape)

torch.Size([1, 101, 33, 59])
